[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t09_survey.ipynb)

# How trigonometry measured Everest

On 10 April 1802, near Madras, William Lambton began measuring a 12-kilometre baseline: the start of the Great
Trigonometrical Survey of India. Half a century later, in 1849–50, J. O. Nicholson sighted Peak XV from six stations
108 to 118 miles away. Their six heights averaged 29,002 feet, and in 1856 the Survey called it most probably the
highest mountain in the world. The Survey's computing office, led by Radhanath Sikdar, turned the angles into
heights (historians still debate who did which step). Today's figure is 8,848.86 m (2020).

The maths: **a tiny angle times a huge distance is a height**, and the curving Earth has to be added back.

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Height from an angle and a distance (flat ground)

Stand a known distance d from a peak and measure the angle α from the horizontal up to its summit. On flat ground,
the height above your eye is **h = d × tan α**. The turning arrow again: tan α is where its line meets the line x = 1,
one unit away, scaled up by d.

In [ ]:
def flat(alpha_deg=10.0, d_km=20.0):
    a = math.radians(alpha_deg)
    h = d_km * math.tan(a)
    fig, ax = plt.subplots(figsize=(11, 4.4))
    ax.plot([0, d_km], [0, 0], color=DIM, lw=2)
    ax.plot([d_km, d_km], [0, h], color=SAGE, lw=6, solid_capstyle="butt")
    ax.plot([0, d_km], [0, h], color=CORAL, lw=2, ls="--")
    ax.fill([d_km - 0.12 * d_km, d_km, d_km + 0.12 * d_km], [0, h, 0], color=DIM, alpha=0.25)
    ax.set_xlim(-0.05 * d_km, 1.2 * d_km); ax.set_ylim(0, max(h * 1.25, 1)); ax.grid(False)
    ax.set_xlabel("km"); ax.set_ylabel("km")
    ax.set_title(f"h = {d_km:.0f} km × tan {alpha_deg:.1f}° = {h:.2f} km  (not to scale vertically)")
    plt.show()

interact(flat, alpha_deg=FloatSlider(value=10, min=0.5, max=40, step=0.1, description="angle (°)"),
         d_km=FloatSlider(value=20, min=1, max=200, step=1, description="distance (km)"));

## 2 · The Earth curves away

Over 108 miles, the Earth's surface drops away below a level line by about **d² ÷ 2R**, where R is the Earth's
radius. That's not small: it's thousands of feet, so a flat-Earth sum would be badly wrong.

In [ ]:
R_FT = 6371.0 * 1000 / 0.3048                     # the Earth's mean radius, 6,371 km, in feet
MILE = 5280.0
def drop_ft(d_miles): return (d_miles * MILE) ** 2 / (2 * R_FT)
for m in (10, 50, 108, 118):
    print(f"{m:4d} miles away: the ground has dropped about {drop_ft(m):6,.0f} ft below a level line")
assert round(drop_ft(108), -2) == 7800

In [ ]:
d = np.linspace(0, 120, 300)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d, drop_ft(d), color=TEAL, lw=3)
ax.plot([108], [drop_ft(108)], "o", color=CORAL, ms=10)
ax.annotate(f"108 miles: {drop_ft(108):,.0f} ft", (108, drop_ft(108)), xytext=(-170, -10), textcoords="offset points", color=CORAL)
ax.set_xlabel("distance (miles)"); ax.set_ylabel("drop below a level line (ft)")
ax.set_title("the curve of the Earth, d² ÷ 2R"); plt.show()

## 3 · Light bends in the air

Air is denser near the ground, so a ray of light from the summit bends slightly downward on its way to you. That
makes the peak look a little higher than it is. Surveyors correct for it with a **refraction coefficient** k; a
typical value is 0.13, which cancels about 13% of the Earth's curve.

In [ ]:
K = 0.13          # a typical refraction coefficient (it changes with the weather)
print(f"at 108 miles, refraction is worth about {K * drop_ft(108):,.0f} ft")

## 4 · Putting it together: how high did Everest look?

Height = d × tan α + (1 − k) × d² ÷ 2R, plus the station's own height. Slide the distance to see the angle Nicholson
would have measured to a summit 29,002 ft high. We take the station to be near sea level. That is an assumption: the
stations were on the plains, a few hundred feet up.

In [ ]:
H = 29_002.0
def elevation(d_miles=108.0):
    dd = d_miles * MILE
    a = math.degrees(math.atan((H - (1 - K) * drop_ft(d_miles)) / dd))
    fig, ax = plt.subplots(figsize=(11, 4.2))
    x = np.linspace(0, d_miles, 300)
    curve = -drop_ft(x) * (1 - K)                    # the ground, falling away, as the observer sees it after refraction
    ax.plot(x, curve, color=DIM, lw=2, label="the ground (Earth's curve, after refraction)")
    ax.plot([d_miles, d_miles], [curve[-1], curve[-1] + H], color=SAGE, lw=6, solid_capstyle="butt", label="Peak XV")
    ax.plot([0, d_miles], [0, curve[-1] + H], color=CORAL, lw=2, ls="--", label=f"line of sight: {a:.2f}° up")
    ax.axhline(0, color=DIM, lw=0.8, ls=":")
    ax.set_xlabel("miles"); ax.set_ylabel("feet"); ax.legend(loc="upper left", fontsize=11)
    ax.set_title(f"from {d_miles:.0f} miles, the summit is only {a:.2f}° above the horizontal"); plt.show()

interact(elevation, d_miles=FloatSlider(value=108, min=60, max=140, step=1, description="distance (mi)"));

In [ ]:
a108 = math.degrees(math.atan((H - (1 - K) * drop_ft(108)) / (108 * MILE)))
print(f"at 108 miles: about {a108:.2f}°")
assert round(a108, 1) == 2.2

**Notice.** A whole mountain fits inside about two degrees. That is why every hundredth of a degree mattered.

## 5 · Six stations, one height

Nicholson's six results, in feet, as reported by Bradford Washburn (American Alpine Club, 1995). Their average is the
famous 29,002.

In [ ]:
STATIONS = {"Jirol": 28_991, "Mirzapur": 29_005, "Joafpati": 29_001, "Ladnia": 28_998, "Harpur": 29_026, "Minai": 28_990}
mean = sum(STATIONS.values()) / len(STATIONS)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(list(STATIONS), list(STATIONS.values()), color=TEAL, width=0.55)
ax.axhline(mean, color=CORAL, lw=2, label=f"average {mean:,.1f} ft")
ax.set_ylim(28_950, 29_040); ax.set_ylabel("feet"); ax.legend(loc="upper left")
ax.set_title("Peak XV from six stations, 1849–50"); plt.show()
print(f"average: {mean:,.1f} ft  →  29,002 ft")
assert round(mean) == 29_002

## 6 · The cost of a tiny error

If the angle is off by a small amount Δα (in radians), the height moves by about **d × sec² α × Δα**. At 108 miles,
a hundredth of a degree is worth about 100 feet. Small angles have to be measured in radians to see this: Δα in
radians is Δα in degrees × π ÷ 180.

In [ ]:
def error(err_deg=0.01, d_miles=108.0):
    dd = d_miles * MILE; a = math.radians(2.2)
    dh = dd * (1 / math.cos(a)) ** 2 * math.radians(err_deg)
    print(f"an angle error of {err_deg:.3f}° at {d_miles:.0f} miles moves the height by about {dh:,.0f} ft")

interact(error, err_deg=FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001, readout_format=".3f", description="error (°)"),
         d_miles=FloatSlider(value=108, min=10, max=140, step=1, description="distance (mi)"));

In [ ]:
dh100 = 108 * MILE * (1 / math.cos(math.radians(2.2))) ** 2 * math.radians(0.01)
assert 95 < dh100 < 105
print(f"0.01° at 108 miles ≈ {dh100:.0f} ft")

## 7 · How they knew the distance: triangulation (beyond the syllabus)

The Survey never walked 108 miles with a tape. It measured one baseline very carefully, then only angles: from each
end of a known side, sight the far point. The triangle's shape fixes everything, and the **sine rule**
(side ÷ sin of opposite angle is the same for all three sides) gives the distances. Chains of such triangles
carried the baseline's accuracy across India. (The sine rule left the CBSE syllabus in 2023; it's here as background.)

In [ ]:
def triangulate(A_deg=62.0, B_deg=70.0, base_km=12.0):
    A, B = math.radians(A_deg), math.radians(B_deg)
    C = math.pi - A - B
    da = base_km * math.sin(B) / math.sin(C)             # from the left end of the baseline to the far point
    P = (da * math.cos(A), da * math.sin(A))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, base_km], [0, 0], color=CORAL, lw=6, solid_capstyle="butt", label=f"baseline {base_km:.0f} km (measured)")
    ax.plot([0, P[0], base_km], [0, P[1], 0], color=TEAL, lw=2.2)
    ax.plot(*P, "o", color=SAGE, ms=10)
    ax.set_aspect("equal"); ax.legend(loc="upper right"); ax.set_xlabel("km")
    ax.set_title(f"angles {A_deg:.0f}° and {B_deg:.0f}° → distance {da:.2f} km")
    plt.show()

interact(triangulate, A_deg=FloatSlider(value=62, min=20, max=85, step=1, description="angle A (°)"),
         B_deg=FloatSlider(value=70, min=20, max=85, step=1, description="angle B (°)"),
         base_km=FloatSlider(value=12, min=5, max=20, step=0.5, description="baseline (km)"));

## 8 · The identity gallery

The facts this measurement leans on.

**tan α = h ÷ d**, from the turning arrow: tan is where its line meets x = 1, so d × tan α is the height.
**1 + tan² α = sec² α**: the error formula's sec² comes from here. **tan α ≈ α for small α in radians**: at 2.2°
the two agree to within 0.05%.

In [ ]:
check_identity(lambda x: 1 + np.tan(x) ** 2, lambda x: 1 / np.cos(x) ** 2, ok=away_from_cos0)
for deg in (0.5, 1, 2.2, 5):
    v = math.radians(deg)
    print(f"{deg:4}°: α = {v:.6f} rad, tan α = {math.tan(v):.6f}, gap {100 * (math.tan(v) - v) / v:.3f}%")

## 9 · On paper: the CBSE way

**Example 1.** From 100 m away, the top of a tower is seen at 30° above the horizontal. How tall is it (above eye
level)?

1. h = d × tan 30° = 100 × (1/√3).
2. **h ≈ 57.7 m.**

In [ ]:
print(f"h = {100 * math.tan(math.radians(30)):.1f} m")

**Example 2.** A hill 1.5 km high is 60 km away. About what angle does it rise above the horizontal (ignore the
Earth's curve)?

1. tan α = 1.5 ÷ 60 = 0.025. For a small angle, α ≈ 0.025 radians.
2. In degrees: 0.025 × 180 ÷ π ≈ **1.43°**.

In [ ]:
print(f"exact: {math.degrees(math.atan(1.5 / 60)):.3f}°,  small-angle: {math.degrees(0.025):.3f}°")

**Try it.**
1. From 50 m away, a building's top is at 60°. How tall is it?
2. At 10 km, how much does an angle error of 0.1° change a measured height?

<details><summary>Answers</summary>

1. 50 × tan 60° = 50√3 ≈ 86.6 m.
2. About 10,000 × 0.1 × π ÷ 180 ≈ 17.5 m (for a small angle, sec² α is almost 1).

</details>

## Check yourself
1. Why must the Earth's curve be added back for a peak 108 miles away?
2. What is 0.01° in radians?
3. What is today's official height of Everest?

<details><summary>Answers</summary>

1. Over that distance the ground drops about 7,800 ft below a level line, a quarter of the mountain's height.
2. About 0.000175 radians (0.01 × π ÷ 180).
3. 8,848.86 m, measured by Nepal and China in 2020.

</details>

## Sources
Bradford Washburn, "Mount Everest: surveying the third pole", American Alpine Club (1995); Royal Geographical
Society, factsheet on measuring Mount Everest; Rajesh Kochhar, The Wire, on Radhanath Sikdar's role. CBSE
Mathematics syllabus 2026–27; NCERT rationalised-content booklet (class 11). All retrieved 2026-10-21. The refraction
coefficient 0.13 is a typical textbook value, not the Survey's own.